# 03_machine_learning

Train and evaluate the classical machine-learning models.

---

*DiaPredict AI — Explainable Diabetes Risk Prediction System.* *Educational / research prototype, not a medical diagnosis.*

## Objectives
- Train Logistic Regression, Random Forest and XGBoost
- Evaluate with cross-validation and a held-out test set
- Compare the models and inspect feature importance

In [ ]:
import sys
from pathlib import Path

# Make the project root importable no matter where the notebook runs.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
print("Root:", PROJECT_ROOT)


### 1. Prepare data

In [ ]:
from backend.app.preprocessing import load_raw_dataset, make_train_test

raw = load_raw_dataset()
X_train, y_train, X_test, y_test, scaler, medians = make_train_test(raw)
print(X_train.shape, X_test.shape)

### 2. Train all three models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced"),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42, n_estimators=300, learning_rate=0.05, max_depth=4),
}
for name, model in models.items():
    model.fit(X_train, y_train)
print("Training complete.")

### 3. Evaluate on the held-out test set

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

results = []
for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba),
    })
pd.DataFrame(results).round(3)

### 4. Cross-validation (5-fold ROC-AUC)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

for name, model in models.items():
    cv = cross_val_score(model, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="roc_auc", n_jobs=-1)
    print(f"{name:20s} CV ROC-AUC: {cv.mean():.4f} +/- {cv.std():.4f}")

### 5. Feature importance

In [ ]:
from backend.app.config import FEATURES

rf = models["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
imp.plot(kind="barh", color="#60a5fa")
plt.title("Random Forest feature importance")
plt.xlabel("Mean decrease in impurity")
plt.show()

### 6. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, model) in zip(axes, models.items()):
    pred = model.predict(X_test)
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
    ax.set_title(f"{name}\nAccuracy {accuracy_score(y_test, pred):.3f}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## Summary

All three models train in seconds and reach ROC-AUC in the 0.78-0.82 range on the test set. Random Forest and Logistic Regression are the strongest here; the full comparison (including the neural network) lives in `05_model_comparison`.